# Low-Rank RNN Paradigm B: LINT on a CTRNN-Trained Mante Task

This notebook demonstrates that the **standard NeuralRNN `Trainer` can be used end-to-end** for the LINT (Low-rank Inference from Neural Trajectories) workflow:

1. We train a conventional **CTRNN** on the Mante context-dependent decision task using `Trainer` + `SupervisedObjective`.
2. We collect the CTRNN's hidden firing-rate trajectories.
3. We fit a **low-rank RNN** to those trajectories *also* using `Trainer` + `SupervisedObjective(regression)`, by setting the low-rank network's readout to the identity during fitting.
4. We plug the CTRNN's readout into the fitted low-rank network and check task accuracy.

The goal is to show that nothing in the LINT protocol requires a custom training loop or changes to the framework: the same generic `Trainer` and `SupervisedObjective` that train CTRNNs on tasks also train low-rank networks on trajectory-reconstruction, once the targets and readout are chosen appropriately.

the current notebook replaces the teacher fullrank model with a standard `CTRNNModel` from the framework and performs *both* training phases with the generic `Trainer`.

**References**
- Valente, A., Ostojic, S., & Pillow, J. W. (2022). *Extracting computational mechanisms from neural data using low-rank RNNs.* NeurIPS.
- Mante, V., Sussillo, D., Shenoy, K. V., & Newsome, W. T. (2013). Context-dependent computation by recurrent dynamics in prefrontal cortex. *Nature*.


## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../src')

import os
import copy
import itertools
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA

# NeuralRNN framework (everything is from the framework)
from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments
from neuralrnn import SupervisedObjective, load_dataset
from neuralrnn.data import CognitiveTaskDataset
from neuralrnn.data.tasks.mante2_task import (
    generate_trials as mante2_trials,
    total_duration, stim_begin, stim_end, response_begin,
    fixation_discrete, ctx_pre_discrete,
    SCALE, SCALE_CTX, STD_DEFAULT,
)
from neuralrnn.train.losses import loss_mse, accuracy_general
from neuralrnn.analysis.linalg_utils import phi_prime, overlap_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

MODEL_DIR = './models/08'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Model directory: {MODEL_DIR}')

plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.titlepad'] = 20
plt.rcParams['axes.labelpad'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 13
plt.rcParams['legend.fontsize'] = 'medium'

### 0.1 Notebook-local helpers

These are the same analysis/plotting utilities used in `08`.


In [ ]:
def generate_mante_data(num_trials, coherences=None, std=None,
                        fraction_validation_trials=0.2, fraction_catch_trials=0.,
                        coh_color_spec=None, coh_motion_spec=None, context_spec=None, seed=None):
    if std is None:
        std = STD_DEFAULT
    inputs, targets, mask, _ = mante2_trials(
        n_trials=num_trials, coherences=coherences, sigma_in=std,
        catch_fraction=fraction_catch_trials,
        coh_color=coh_color_spec, coh_motion=coh_motion_spec,
        context=context_spec, seed=seed,
    )
    if fraction_validation_trials > 0:
        split = num_trials - int(num_trials * fraction_validation_trials)
        return (inputs[:split], targets[:split], mask[:split],
                inputs[split:], targets[split:], mask[split:])
    return inputs, targets, mask


def generate_mante_data_from_conditions(coherences_A, coherences_B, contexts, std=0, seed=None):
    if seed is not None:
        np.random.seed(seed)
    num_trials = coherences_A.shape[0]
    inputs_sensory = std * torch.randn((num_trials, total_duration, 2), dtype=torch.float32)
    inputs_context = torch.zeros((num_trials, total_duration, 2))
    inputs = torch.cat([inputs_sensory, inputs_context], dim=2)
    targets = torch.zeros((num_trials, total_duration, 1), dtype=torch.float32)
    mask = torch.zeros((num_trials, total_duration, 1), dtype=torch.float32)
    for i in range(num_trials):
        inputs[i, stim_begin:stim_end, 0] += coherences_A[i] * SCALE
        inputs[i, stim_begin:stim_end, 1] += coherences_B[i] * SCALE
        if contexts[i] == 1:
            inputs[i, fixation_discrete:response_begin, 2] = 1.0 * SCALE_CTX
            targets[i, response_begin:, 0] = 1.0 if coherences_A[i] > 0 else -1.0
        elif contexts[i] == -1:
            inputs[i, fixation_discrete:response_begin, 3] = 1.0 * SCALE_CTX
            targets[i, response_begin:, 0] = 1.0 if coherences_B[i] > 0 else -1.0
        mask[i, response_begin:, 0] = 1.0
    return inputs, targets, mask


def test_mante(net, x, y, mask):
    net.eval()
    with torch.no_grad():
        output = net(x)
        # net(x) returns DynamicsModelOutput for both CTRNN and LowrankRNNModel
        y_hat = output.outputs if hasattr(output, 'outputs') else output
        loss = loss_mse(y_hat, y, mask)
        acc = accuracy_general(y_hat, y, mask)
    return loss.item(), acc.item()


def psychometric_matrices(net, coherences=None, cmap='gray', figsize=(4, 8), axes=None, colorbar=False):
    n_trials = 10
    if coherences is None:
        coherences = np.arange(-5, 6, 2)
    if axes is None:
        fig, axes = plt.subplots(2, 1, figsize=figsize)
    for ctx in (0, 1):
        mat = np.zeros((len(coherences), len(coherences)))
        for i, coh1 in enumerate(coherences):
            for j, coh2 in enumerate(coherences):
                inputs_sensory = STD_DEFAULT * torch.randn((n_trials, total_duration, 2), dtype=torch.float32)
                inputs_context = torch.zeros((n_trials, total_duration, 2))
                inputs = torch.cat([inputs_sensory, inputs_context], dim=2)
                inputs[:, stim_begin:stim_end, 0] += coh1 * SCALE
                inputs[:, stim_begin:stim_end, 1] += coh2 * SCALE
                inputs[:, fixation_discrete:response_begin, 2 + ctx] = 1.0 * SCALE_CTX
                with torch.no_grad():
                    output = net(inputs)
                    y_hat = output.outputs if hasattr(output, 'outputs') else output
                    decisions = torch.sign(y_hat[:, response_begin:, :].mean(dim=1).squeeze())
                mat[len(coherences) - j - 1, i] = decisions.mean().item()
        im = axes[ctx].matshow(mat, cmap=cmap, vmin=-1, vmax=1)
        axes[ctx].set_xticks([])
        axes[ctx].set_yticks([])
    if colorbar:
        fig = axes[0].figure
        ax_cbar = fig.add_axes([0.9, 0.3, 0.04, 0.4])
        fig.colorbar(im, cax=ax_cbar)
    return axes


def inactivate_pop(net, pop):
    # Inactivate a population, returning a new network with those neurons removed.
    from neuralrnn.models.ctrnn.modeling_ctrnn import CTRNNModel
    from neuralrnn.models.lowrank.modeling_lowrank import LowrankRNNModel
    pop = np.asarray(pop)
    keep = ~pop
    size = int(np.sum(keep))

    if isinstance(net, CTRNNModel):
        cfg = copy.deepcopy(net.config)
        cfg.latent_dim = size
        net2 = CTRNNModel(cfg).to(net.h0.device)
        with torch.no_grad():
            # input2h.weight: (latent_dim, input_dim)  -> keep rows
            net2.input2h.weight.copy_(net.input2h.weight[keep, :])
            net2.input2h.bias.copy_(net.input2h.bias[keep])
            # h2h.weight: (latent_dim, latent_dim)
            net2.h2h.weight.copy_(net.h2h.weight[np.ix_(keep, keep)])
            net2.h2h.bias.copy_(net.h2h.bias[keep])
            # readout_layer.weight: (output_dim, latent_dim) -> keep columns
            net2.readout_layer.weight.copy_(net.readout_layer.weight[:, keep])
            net2.readout_layer.bias.copy_(net.readout_layer.bias)
            if cfg.trainable_h0:
                net2.h0.copy_(net.h0[keep])
            if net.dale_mask is not None:
                net2.dale_mask.copy_(net.dale_mask[np.ix_(keep, keep)])
        return net2

    if isinstance(net, LowrankRNNModel):
        cfg = copy.deepcopy(net.config)
        cfg.latent_dim = size
        net2 = LowrankRNNModel(cfg).to(net.m.device)
        with torch.no_grad():
            net2.m.copy_(net.m[keep])
            net2.n.copy_(net.n[keep])
            net2.wi.copy_(net.wi[:, keep])
            net2.wo.copy_(net.wo[keep])
            net2.si.copy_(net.si)
            net2.so.copy_(net.so)
            net2.b.copy_(net.b[keep])
            net2.h0.copy_(net.h0[keep])
        net2._define_proxy_parameters()
        return net2

    raise TypeError(f'inactivate_pop does not support {type(net)}')


def center_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_position('zero')
    ax.spines['left'].set_position('zero')
    ax.set(xticks=[], yticks=[])


def pop_scatter_linreg(vec1, vec2, pops, n_pops=None, linreg=True,
                       colors=('blue', 'red', 'green', 'gray'),
                       figsize=(5, 5), ax=None, dotsize=4, ls=None, lw=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    center_axes(ax)
    xmax = max(abs(vec1.min()), vec1.max())
    xmin = -xmax
    ax.set_xlim(xmin - .1 * (xmax - xmin), xmax + .1 * (xmax - xmin))
    ymax = max(abs(vec2.min()), vec2.max())
    ymin = -ymax
    ax.set_ylim(ymin - .1 * (ymax - ymin), ymax + .1 * (ymax - ymin))
    xs = np.linspace(xmin - .1 * (xmax - xmin), xmax + .1 * (xmax - xmin), 100)
    if n_pops is None:
        n_pops = np.unique(pops).shape[0]
    for i in range(n_pops):
        ax.scatter(vec1[pops == i], vec2[pops == i], color=colors[i], s=dotsize)
        if linreg:
            slope, intercept, _, _, _ = stats.linregress(vec1[pops == i], vec2[pops == i])
            print(f"pop {i}: slope={slope:.2f}, intercept={intercept:.2f}")
            ax.plot(xs, slope * xs + intercept, color=colors[i], zorder=-1, ls=ls, lw=lw)
    ax.set_xticks([])
    ax.set_yticks([])
    return ax


def overlap_matrix2(vecs1, vecs2, norm='overlap', abs=False):
    size = len(vecs1[0])
    ov = np.zeros((len(vecs1), len(vecs2)))
    for i, v1 in enumerate(vecs1):
        for j, v2 in enumerate(vecs2):
            if norm == 'overlap':
                ov[i, j] = v1 @ v2 / size
            else:
                ov[i, j] = v1 @ v2 / np.sqrt((v1 @ v1) * (v2 @ v2))
    if abs:
        ov = np.abs(ov)
    return ov


def plot_ovmat(ov, cmap=None, figsize=None, cbar=True, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    if cmap is None:
        bound = max(abs(ov.min()), abs(ov.max()))
        cmap = sns.diverging_palette(220, 10, sep=10, as_cmap=True)
        sns.heatmap(ov, ax=ax, cmap=cmap, vmin=-bound, vmax=bound, cbar=cbar)
    elif cmap == 'abs':
        cmap = sns.color_palette("light:firebrick", as_cmap=True)
        sns.heatmap(ov, ax=ax, cmap=cmap, cbar=cbar)
    return ax


def plot_projections(trials, axis1, axis2, colors, lab1, lab2, trials_other=None, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    for i, tr in enumerate(trials):
        ax.plot((tr @ axis1).mean(axis=0), (tr @ axis2).mean(axis=0), c=colors[i], lw=4)
        if trials_other is not None:
            ax.plot((trials_other[i] @ axis1).mean(axis=0), (trials_other[i] @ axis2).mean(axis=0),
                    c=colors[i], ls=':', lw=4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel(lab1)
    ax.set_ylabel(lab2)
    return ax


def plot_gains(traj, z, colors, xs, title, show_xlabel=False, show_ylabel=False, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))
    ys = np.tanh(xs)
    kde1 = KernelDensity(kernel='gaussian', bandwidth=.1).fit(
        traj[:, :ctx_pre_discrete, z == 1].mean(axis=(0, 1)).reshape((-1, 1)))
    ys1 = np.exp(kde1.score_samples(xs.reshape((-1, 1))))
    ys1 = ys1 / np.sum(ys1) * 15
    kde2 = KernelDensity(kernel='gaussian', bandwidth=.1).fit(
        traj[:, :ctx_pre_discrete, z == 2].mean(axis=(0, 1)).reshape((-1, 1)))
    ys2 = np.exp(kde2.score_samples(xs.reshape((-1, 1))))
    ys2 = ys2 / np.sum(ys2) * 15

    ax.plot(xs, ys, c='k', lw=3)
    ax.plot(xs, ys1 + 1.1, c=colors[1], lw=2.5)
    ax.plot(xs, ys2 + 1.1, c=colors[2], lw=2.5)
    ax.set_ylim(-1.1, 3.4)
    ax.spines['left'].set_bounds([-1, 1])
    ax.set_yticks([-1, 0, 1])
    if not show_xlabel:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel('$x_i$')
    if not show_ylabel:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel('$\\phi(x_i)$', position=(0, .3))
    ax.set_title(title)
    return ax

print('Helpers defined.')


## 1. Train a CTRNN teacher on the Mante task

We use the framework's standard `Trainer` and `SupervisedObjective` for supervised regression. The CTRNN is a full-rank, vanilla recurrent network with `latent_dim=512`. Hyper-parameters (`dt`, `tau`, `sigma_rec`, learning rate, etc.) are chosen so that the network learns the task to high accuracy in a reasonable number of steps.

Because `TrainingArguments.device` defaults to `"cpu"`, we explicitly pass the same `device` used for evaluation.


In [ ]:
size = 512
alpha_ctrnn = 0.2  # dt/tau = 0.2 corresponds to dt=20ms, tau=100ms
noise_std = 5e-2

print('Generating Mante training data...')
x_train, y_train, mask_train, x_val, y_val, mask_val = generate_mante_data(
    1000, fraction_validation_trials=0.2, seed=42)
print(f'Train: {x_train.shape}, Val: {x_val.shape}')

ctrnn_path = os.path.join(MODEL_DIR, 'ctrnn_mante_teacher.pt')
if os.path.exists(ctrnn_path):
    print(f'Loading CTRNN checkpoint from {ctrnn_path}')
    cfg_ctrnn = AutoConfig.for_model('ctrnn',
        input_dim=4, latent_dim=size, output_dim=1,
        dt=20, tau=100, activation='tanh', sigma_rec=noise_std,
        trainable_h0=False, nonlinearity_mode='pre_activation')
    net_ctrnn = AutoModel.from_config(cfg_ctrnn).to(device)
    net_ctrnn.load_state_dict(torch.load(ctrnn_path, map_location=device))
else:
    print('Training CTRNN teacher on Mante (this may take a few minutes)...')
    cfg_ctrnn = AutoConfig.for_model('ctrnn',
        input_dim=4, latent_dim=size, output_dim=1,
        dt=20, tau=100, activation='tanh', sigma_rec=noise_std,
        trainable_h0=False, nonlinearity_mode='pre_activation')
    net_ctrnn = AutoModel.from_config(cfg_ctrnn).to(device)

    ds_train = CognitiveTaskDataset(x_train, y_train, mask_train, conditions=[], batch_size=32)
    objective = SupervisedObjective(task_type='regression')
    args = TrainingArguments(
        max_steps=1500, learning_rate=1e-3, grad_clip_norm=1.0,
        keep_best=True, log_every=100, device=str(device))
    Trainer(net_ctrnn, ds_train, objective, args).train()
    torch.save(net_ctrnn.state_dict(), ctrnn_path)

loss_ctrnn, acc_ctrnn = test_mante(net_ctrnn, x_val.to(device), y_val.to(device), mask_val.to(device))
print(f'CTRNN teacher - val loss: {loss_ctrnn:.4f}, accuracy: {acc_ctrnn:.4f}')


## 2. Collect CTRNN firing-rate trajectories

The CTRNN's `forward(..., return_states=True)` returns `DynamicsModelOutput(outputs=..., states=...)`. In the CTRNN recurrence the state `z_t` is already the post-nonlinearity activity, so `states` can be treated directly as the firing-rate trajectories that the low-rank network should reproduce.


In [ ]:
net_ctrnn.eval()
with torch.no_grad():
    out_train = net_ctrnn(x_train.to(device), return_states=True)
    target_rates_train = out_train.states.detach().cpu()
    out_val = net_ctrnn(x_val.to(device), return_states=True)
    target_rates_val = out_val.states.detach().cpu()

print(f'Teacher firing-rate trajectories: train={tuple(target_rates_train.shape)}, val={tuple(target_rates_val.shape)}')

# Mask for trajectory fitting: every timestep counts
mask_rates = torch.ones(x_train.shape[0], x_train.shape[1], 1)


## 3. LINT: fit low-rank RNNs to the CTRNN trajectories with the standard `Trainer`

This is the central section. For each rank $r$:

1. Build a `LowrankRNNModel` with `output_dim = latent_dim`.
2. Fix the readout to the identity (`wo = N * I`, `so = ones`), so the network's outputs are exactly its reconstructed firing rates.
3. Construct a `CognitiveTaskDataset` whose targets are the CTRNN firing-rate trajectories and whose mask is all ones.
4. Train with `Trainer` + `SupervisedObjective(task_type='regression')` — the exact same combination used for task training.
5. After fitting, copy the low-rank factors into a second network with `output_dim=1`, load the CTRNN's readout, and evaluate task accuracy.

The only difference from a standard supervised training run is the choice of targets (teacher trajectories) and the temporary identity readout.


In [ ]:
rank_max = 1  # set to 14 to reproduce the full Valente et al. (2022) scan

r2s_fit = []
losses_fit = []
accs_fit = []

for rank in range(1, rank_max + 1):
    print(f'\n=== Fitting rank {rank} ===')
    lr_path = os.path.join(MODEL_DIR, f'lint_ctrnn_matched_r{rank}.pt')
    if os.path.exists(lr_path):
        print(f'Loading rank-{rank} checkpoint from {lr_path}')
        cfg_fit = AutoConfig.for_model('lowrank_rnn',
            input_dim=4, latent_dim=size, output_dim=size,
            rank=rank, alpha=alpha_ctrnn, sigma_rec=noise_std,
            scale_by_hidden_size=True, train_wi=True, train_wo=False,
            activation='tanh', output_activation='tanh')
        net_lr = AutoModel.from_config(cfg_fit).to(device)
        net_lr.load_state_dict(torch.load(lr_path, map_location=device))
    else:
        cfg_fit = AutoConfig.for_model('lowrank_rnn',
            input_dim=4, latent_dim=size, output_dim=size,
            rank=rank, alpha=alpha_ctrnn, sigma_rec=noise_std,
            scale_by_hidden_size=True, train_wi=True, train_wo=False,
            activation='tanh', output_activation='tanh')
        net_lr = AutoModel.from_config(cfg_fit).to(device)
        with torch.no_grad():
            net_lr.wo.set_(size * torch.eye(size, device=device))
            net_lr.so.set_(torch.ones(size, device=device))
        net_lr.wo.requires_grad = False
        net_lr.so.requires_grad = False

        ds_lint = CognitiveTaskDataset(x_train, target_rates_train, mask_rates,
                                       conditions=[], batch_size=32)
        objective_lint = SupervisedObjective(task_type='regression')
        args_lint = TrainingArguments(
            max_steps=4000, learning_rate=2e-2, grad_clip_norm=1.0,
            keep_best=True, log_every=200, device=str(device))
        Trainer(net_lr, ds_lint, objective_lint, args_lint).train()
        torch.save(net_lr.state_dict(), lr_path)

    # Validation R2 between reconstructed and teacher firing rates
    net_lr.eval()
    with torch.no_grad():
        pred_val = net_lr(x_val.to(device)).outputs.detach().cpu()
        r2 = r2_score(target_rates_val.numpy().ravel(), pred_val.numpy().ravel())

    # Plug in the CTRNN readout and evaluate task performance
    cfg_final = AutoConfig.for_model('lowrank_rnn',
        input_dim=4, latent_dim=size, output_dim=1,
        rank=rank, alpha=alpha_ctrnn, sigma_rec=noise_std,
        scale_by_hidden_size=True, train_wi=True, train_wo=True,
        activation='tanh', output_activation='tanh')
    net_lr_final = AutoModel.from_config(cfg_final).to(device)
    with torch.no_grad():
        net_lr_final.m.copy_(net_lr.m)
        net_lr_final.n.copy_(net_lr.n)
        net_lr_final.wi.copy_(net_lr.wi)
        net_lr_final.si.copy_(net_lr.si)
        # CTRNN readout weight shape is (output_dim=1, latent_dim=size); transpose to (size, 1)
        net_lr_final.wo.copy_(net_ctrnn.readout_layer.weight.data.t())
        net_lr_final.so.set_(torch.tensor([1.0 * size], device=device))
        net_lr_final.b.copy_(net_lr.b)
        net_lr_final.h0.copy_(net_lr.h0)
    net_lr_final._define_proxy_parameters()

    loss, acc = test_mante(net_lr_final, x_val.to(device), y_val.to(device), mask_val.to(device))
    r2s_fit.append(r2)
    losses_fit.append(loss)
    accs_fit.append(acc)
    print(f'rank {rank:2d}: R2={r2:.3f}, loss={loss:.3f}, acc={acc:.3f}')

# ---- Truncated-SVD baseline (rank 1 only, since rank_max=1) ----
print('\n=== Truncated-SVD baseline ===')
J = net_ctrnn.h2h.weight.data.detach().cpu().numpy()
u, s, v = np.linalg.svd(J)
rank = 1
J_rec = (u[:, :rank] * s[:rank]) @ v[:rank, :]
net_trunc = copy.deepcopy(net_ctrnn)
net_trunc.h2h.weight.data.copy_(torch.from_numpy(J_rec).to(device))
net_trunc.eval()
with torch.no_grad():
    out_t = net_ctrnn(x_val.to(device), return_states=True)
    traj_fr = out_t.states.detach().cpu().numpy()
    out_trunc = net_trunc(x_val.to(device), return_states=True)
    traj_trunc = out_trunc.states.detach().cpu().numpy()
    r2_trunc = r2_score(traj_fr.ravel(), traj_trunc.ravel())
loss_trunc, acc_trunc = test_mante(net_trunc, x_val.to(device), y_val.to(device), mask_val.to(device))
print(f'rank {rank:2d}: R2={r2_trunc:.3f}, loss={loss_trunc:.3f}, acc={acc_trunc:.3f}')


## 4. Rank scan: $R^2$ and accuracy vs. rank

We expect the low-rank fit to reach high state-space $R^2$ already at low rank, and for task accuracy after readout transfer to approach the teacher's accuracy.


### 4.1 Truncated-SVD baseline

As a structural baseline, we compute the rank-1 truncated SVD of the CTRNN recurrent weight
matrix `h2h.weight`, replace the recurrent weights with this rank-1 approximation, and measure
how much of the state-space variance and task performance is preserved *without* retraining.
This is reported in the rank-scan plot above as the red square.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax.plot(np.arange(1, rank_max + 1), r2s_fit, marker='o', c='tab:blue', lw=3, markersize=9, label='LINT')
if rank_max == 1:
    ax.scatter([1], [r2_trunc], marker='s', c='firebrick', s=100, zorder=5, label='Truncated SVD')
ax.set_ylabel('State-space $R^2$')
ax.set_xlabel('Rank')
ax.set_ylim(0, 1)
ax.axhline(1, ls='--', c='gray')
ax.legend()

ax = axes[1]
ax.plot(np.arange(1, rank_max + 1), accs_fit, marker='o', c='tab:blue', lw=3, markersize=9, label='LINT')
if rank_max == 1:
    ax.scatter([1], [acc_trunc], marker='s', c='firebrick', s=100, zorder=5, label='Truncated SVD')
ax.axhline(acc_ctrnn, c='gray', ls='--', label=f'Teacher acc={acc_ctrnn:.3f}')
ax.set_ylabel('Accuracy')
ax.set_xlabel('Rank')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()


## 5. Build the rank-1 fitted network for downstream analysis

The rank scan shows the rank-1 LINT network's performance. We extract its connectivity vectors,
apply the reference sign convention (`m ← -m`, `n ← -n`), and build a clean final network
with the CTRNN readout. All subsequent analyses compare this fitted low-rank network with the
CTRNN teacher.


In [ ]:
rank = 1
lr_fit_path = os.path.join(MODEL_DIR, f'lint_ctrnn_matched_r{rank}.pt')
cfg_lr_fit = AutoConfig.for_model('lowrank_rnn',
    input_dim=4, latent_dim=size, output_dim=size,
    rank=rank, alpha=alpha_ctrnn, sigma_rec=noise_std,
    scale_by_hidden_size=True, train_wi=True, train_wo=False,
    activation='tanh', output_activation='tanh')
net_lr_fit = AutoModel.from_config(cfg_lr_fit).to(device)
net_lr_fit.load_state_dict(torch.load(lr_fit_path, map_location=device))

m = net_lr_fit.m.squeeze().detach().cpu().numpy()
n = net_lr_fit.n.squeeze().detach().cpu().numpy()
wi1 = net_lr_fit.wi[0].detach().cpu().numpy()
wi2 = net_lr_fit.wi[1].detach().cpu().numpy()
wi_ctx1 = net_lr_fit.wi[2].detach().cpu().numpy()
wi_ctx2 = net_lr_fit.wi[3].detach().cpu().numpy()
wo = net_ctrnn.readout_layer.weight.data.squeeze().detach().cpu().numpy()

# Reference sign convention
m = -m
n = -n

cfg_lr_final = AutoConfig.for_model('lowrank_rnn',
    input_dim=4, latent_dim=size, output_dim=1,
    rank=rank, alpha=alpha_ctrnn, sigma_rec=noise_std,
    scale_by_hidden_size=True, train_wi=True, train_wo=True,
    activation='tanh', output_activation='tanh')
net_lr_final = AutoModel.from_config(cfg_lr_final).to(device)
with torch.no_grad():
    net_lr_final.m.set_(torch.from_numpy(m.reshape(-1, 1)).to(device))
    net_lr_final.n.set_(torch.from_numpy(n.reshape(-1, 1)).to(device))
    net_lr_final.wi.copy_(net_lr_fit.wi)
    net_lr_final.si.copy_(net_lr_fit.si)
    net_lr_final.wo.copy_(net_ctrnn.readout_layer.weight.data.t())
    net_lr_final.so.set_(torch.tensor([1.0 * size], device=device))
    net_lr_final.b.copy_(net_lr_fit.b)
    net_lr_final.h0.copy_(net_lr_fit.h0)
net_lr_final._define_proxy_parameters()

loss_lr, acc_lr = test_mante(net_lr_final, x_val.to(device), y_val.to(device), mask_val.to(device))
print(f'Validation accuracy: CTRNN={acc_ctrnn:.3f}, LR={acc_lr:.3f}')


## 6. Population clustering and connectivity overlap

Following the same logic as `08`, we label context-selective populations by the magnitude of their context-cue input weights, then compare the fitted low-rank connectivity vectors with the CTRNN teacher's input/output weights.


In [ ]:
z = np.zeros(size)
z1 = np.argsort(np.abs(wi_ctx1))[::-1][:100]
z[z1] = 1
z2 = np.argsort(np.abs(wi_ctx2))[::-1][:100]
z[z2] = 2
z = np.where(z == 3, 1, z)
colors = ['dimgray', 'seagreen', 'rebeccapurple']

_, counts = np.unique(z, return_counts=True)
fig, ax = plt.subplots(figsize=(4, 4))
ax.bar([0, 1, 2], counts, color=colors)
ax.set_ylabel('# units')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Non-selective', 'Ctx A', 'Ctx B'])
ax.set_title('Population sizes')
plt.tight_layout()
plt.show()


In [ ]:
wi1_o = net_ctrnn.input2h.weight[:, 0].detach().cpu().numpy()
wi2_o = net_ctrnn.input2h.weight[:, 1].detach().cpu().numpy()
wi_ctx1_o = net_ctrnn.input2h.weight[:, 2].detach().cpu().numpy()
wi_ctx2_o = net_ctrnn.input2h.weight[:, 3].detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pop_scatter_linreg(wi_ctx1, wi_ctx2, z, linreg=False, dotsize=20, colors=colors, ax=axes[0])
axes[0].set_title('Context inputs (fitted LR)')
pop_scatter_linreg(wi_ctx1_o, wi_ctx2_o, z, linreg=False, dotsize=20, colors=colors, ax=axes[1])
axes[1].set_title('Context inputs (CTRNN teacher)')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
pop_scatter_linreg(n, wi1, z, dotsize=9, colors=colors, lw=2, ls='dashed', ax=axes[0, 0])
axes[0, 0].set_title('$n$ vs $I^A$')
pop_scatter_linreg(n, wi2, z, dotsize=9, colors=colors, lw=2, ls='dashed', ax=axes[0, 1])
axes[0, 1].set_title('$n$ vs $I^B$')
pop_scatter_linreg(n, m, z, dotsize=9, colors=colors, lw=2, ls='dashed', ax=axes[1, 0])
axes[1, 0].set_title('$n$ vs $m$')
pop_scatter_linreg(m, wo, z, dotsize=9, colors=colors, lw=2, ls='dashed', ax=axes[1, 1])
axes[1, 1].set_title('$m$ vs $w^{out}$')
plt.tight_layout()
plt.show()


### 6.1 Connectivity overlap matrix (SVD vs. fitted)

We compare the fitted rank-1 connectivity vectors with the dominant singular vectors of the
CTRNN recurrent weight matrix. The heatmap shows absolute cosine overlap between the teacher's
input vectors / top SVD modes / output vector and the fitted low-rank vectors.


In [ ]:
wrec = net_ctrnn.h2h.weight.data.detach().cpu().numpy()
u, s, v = np.linalg.svd(wrec)
ms_orig = [u[:, i] for i in range(1, 8)]
ns_orig = [v[i, :] for i in range(1, 8)]
wo_o = net_ctrnn.readout_layer.weight.data.squeeze().detach().cpu().numpy()

vecs_or = [wi1_o, wi2_o, wi_ctx1_o, wi_ctx2_o] + ms_orig + ns_orig + [wo_o]
vecs_fit = [wi1, wi2, wi_ctx1, wi_ctx2, m, n, wo]
ov = overlap_matrix2(vecs_or, vecs_fit, norm='l2', abs=True)

fig, ax = plt.subplots(figsize=(5, 10))
plot_ovmat(ov, cmap='abs', figsize=(5, 10), ax=ax)
ax.set_xlabel('Fitted')
ax.set_ylabel('Original (SVD)')
ax.set_xticks(np.arange(0.5, 7, 1))
ax.set_xticklabels(['wi1', 'wi2', 'ct1', 'ct2', 'm', 'n', 'wo'])
ax.set_yticks(np.arange(0.5, 19, 1))
ax.set_yticklabels(['wi1', 'wi2', 'ct1', 'ct2'] +
                   [f'm{i}' for i in range(1, 8)] + [f'n{i}' for i in range(1, 8)] + ['wo'])
ax.set_title('Connectivity overlap')
plt.tight_layout()
plt.show()


## 7. Dynamics Analysis

We compare the latent dynamics of the CTRNN teacher and the rank-1 LINT network. First, we fit
PCA on the CTRNN firing-rate trajectories and plot the cumulative explained variance; this
quantifies the intrinsic dimensionality of the teacher's dynamics. Second, we project both
networks onto the top-$k$ PCs and compute the $R^2$ between the projected trajectories as a
function of $k$.


## 8. TDR (Targeted Dimensionality Reduction)

Targeted Dimensionality Reduction (TDR) finds task-relevant axes by regressing the population
activity onto task variables: motion coherence, color coherence, context, and choice. We generate
8 condition-averaged trials (all combinations of `motion ∈ {-8, 8}`, `color ∈ {-4, 4}`,
`context ∈ {-1, 1}`), fit a linear regression from the four task variables (plus choice) to the
population activity at each time step, and identify the time point at which each regression
weight has maximal norm. We then orthogonalize the four axes via QR decomposition to obtain a
set of interpretable axes: choice, motion, color, and context.

The CTRNN teacher already returns post-nonlinearity firing rates. For the fitted low-rank network
we take `tanh(states)` so that both networks are compared in firing-rate space, matching the
analysis convention in `08`.

The projection plots show condition-averaged trajectories in the TDR axes (solid = CTRNN,
dotted = fitted). The connectivity-axis projections below show the same trajectories projected
onto the LINT-inferred vectors ($m$, $I^A$, $I^B$), demonstrating that the low-rank connectivity
vectors align with the task variables uncovered by TDR.


In [ ]:
x_train_all, y_train_all, mask_train_all = generate_mante_data(
    1000, fraction_validation_trials=0, seed=42)

net_ctrnn.eval()
net_lr_final.eval()
with torch.no_grad():
    out_t = net_ctrnn(x_train_all.to(device), return_states=True)
    traj1 = out_t.states.detach().cpu().numpy()
    out_l = net_lr_final(x_train_all.to(device), return_states=True)
    traj2 = torch.tanh(out_l.states).detach().cpu().numpy()

traj1_flat = traj1.reshape(-1, traj1.shape[-1])
pca1 = PCA().fit(traj1_flat)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, 21), np.cumsum(pca1.explained_variance_ratio_)[:20], lw=3)
ax.set_xlabel('PC rank')
ax.set_ylabel('Cum. explained variance')
ax.set_title('PCA cumulative variance (CTRNN teacher)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
r2s_components = []
for pc_rank in range(1, 100):
    lowd_space = pca1.components_[:pc_rank]
    traj1_proj = traj1_flat @ lowd_space.T
    traj2_flat = traj2.reshape(-1, traj2.shape[-1])
    traj2_proj = traj2_flat @ lowd_space.T
    r2 = r2_score(traj1_proj.ravel(), traj2_proj.ravel())
    r2s_components.append(r2)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, 100), r2s_components, lw=3)
ax.set_ylim(0, 1)
ax.set_ylabel('$R^2$ of fitted vs original')
ax.set_xlabel('Kept PCs')
ax.axhline(r2s_fit[0], c='gray', ls='--', label=f'Rank-1 fit $R^2$={r2s_fit[0]:.3f}')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
conditions = list(itertools.product([-8, 8], [-4, 4], [-1, 1]))
conditions = np.array(conditions)
motion = conditions[:, 0]
color = conditions[:, 1]
context = conditions[:, 2]
choice = np.where(context == 1, np.where(motion > 0, 1, -1),
                  np.where(color > 0, 1, -1))
conditions_full = np.append(conditions, choice[:, np.newaxis], axis=1)

x_tdr, _, _ = generate_mante_data_from_conditions(conditions[:, 0], conditions[:, 1], conditions[:, 2], std=0)

net_ctrnn.eval()
net_lr_final.eval()
with torch.no_grad():
    out_t = net_ctrnn(x_tdr.to(device), return_states=True)
    traj = out_t.states.detach().cpu().numpy()
    out_l = net_lr_final(x_tdr.to(device), return_states=True)
    traj2 = torch.tanh(out_l.states).detach().cpu().numpy()

ntime = traj.shape[1]
traj_percond = traj.reshape((8, -1))
linmodel = LinearRegression(fit_intercept=True)
linmodel.fit(conditions_full, traj_percond)
betas = linmodel.coef_.T.reshape((4, ntime, -1))
intercepts = linmodel.intercept_.T.reshape((ntime, -1))
betas = np.concatenate([betas, intercepts[np.newaxis, :, :]], axis=0)

tmaxes = []
for i in range(5):
    norms = np.linalg.norm(betas[i], axis=1)
    tmaxes.append(np.argmax(norms))

beta_motion = betas[0, tmaxes[0]]
beta_color = betas[1, tmaxes[1]]
beta_context = betas[2, tmaxes[2]]
beta_choice = betas[3, tmaxes[3]]

Bmat = np.vstack([beta_choice, beta_motion, beta_color, beta_context]).T
BmatQ, _ = np.linalg.qr(Bmat)
beta_choice = BmatQ[:, 0]
beta_motion = BmatQ[:, 1]
beta_color = BmatQ[:, 2]
beta_context = BmatQ[:, 3]

print('TDR axes computed.')


In [ ]:
_trials = [traj[(context == 1) & (choice > 0)], traj[(context == 1) & (choice < 0)]]
_trials_other = [traj2[(context == 1) & (choice > 0)], traj2[(context == 1) & (choice < 0)]]
cmap = matplotlib.cm.get_cmap('bwr')
colors2 = [cmap(0), cmap(255)]
ax = plot_projections(_trials, beta_choice, -beta_motion, colors2, 'Choice axis', 'Stim. A axis', _trials_other)
ax.set_title('Context 1: Choice vs Motion\n(solid=CTRNN, dotted=fitted LR)')
plt.show()


In [ ]:
_trials = [traj[(context == -1) & (choice > 0)], traj[(context == -1) & (choice < 0)]]
_trials_other = [traj2[(context == -1) & (choice > 0)], traj2[(context == -1) & (choice < 0)]]
cmap = matplotlib.cm.get_cmap('PiYG')
colors2 = [cmap(0), cmap(255)]
plot_projections(_trials, beta_choice, beta_color, colors2, 'Choice axis', 'Stim. B axis', _trials_other)
plt.title('Context 2: Choice vs Color')
plt.show()


In [ ]:
_trials = [traj[(context == 1) & (choice > 0) & (color > 0)],
           traj[(context == 1) & (choice > 0) & (color < 0)],
           traj[(context == 1) & (choice < 0) & (color > 0)],
           traj[(context == 1) & (choice < 0) & (color < 0)]]
_trials_other = [traj2[(context == 1) & (choice > 0) & (color > 0)],
                 traj2[(context == 1) & (choice > 0) & (color < 0)],
                 traj2[(context == 1) & (choice < 0) & (color > 0)],
                 traj2[(context == 1) & (choice < 0) & (color < 0)]]
cmap = matplotlib.cm.get_cmap('PiYG')
colors4 = [cmap(0), cmap(255), cmap(0), cmap(255)]
plot_projections(_trials, beta_choice, beta_color, colors4, 'Choice axis', 'Stim. B axis', _trials_other)
plt.title('Context 1: Choice vs Color (solid=CTRNN, dotted=fitted LR)')
plt.show()


In [ ]:
_trials = [traj[(context == -1) & (choice > 0) & (motion > 0)],
           traj[(context == -1) & (choice > 0) & (motion < 0)],
           traj[(context == -1) & (choice < 0) & (motion > 0)],
           traj[(context == -1) & (choice < 0) & (motion < 0)]]
_trials_other = [traj2[(context == -1) & (choice > 0) & (motion > 0)],
                 traj2[(context == -1) & (choice > 0) & (motion < 0)],
                 traj2[(context == -1) & (choice < 0) & (motion > 0)],
                 traj2[(context == -1) & (choice < 0) & (motion < 0)]]
cmap = matplotlib.cm.get_cmap('bwr')
colors4 = [cmap(0), cmap(255), cmap(0), cmap(255)]
plot_projections(_trials, beta_choice, -beta_motion, colors4, 'Choice axis', 'Stim. A axis', _trials_other)
plt.title('Context 2: Choice vs Motion (solid=CTRNN, dotted=fitted LR)')
plt.show()


### 8.1 Connectivity-axis projections

The same condition-averaged trajectories can be projected onto the connectivity axes
uncovered by LINT ($m$, $I^A$, $I^B$). If the low-rank vectors are aligned with the task
variables, these projections should resemble the TDR plots above.


In [ ]:
_trials = [traj[(context == 1) & (choice > 0)], traj[(context == 1) & (choice < 0)]]
_trials_other = [traj2[(context == 1) & (choice > 0)], traj2[(context == 1) & (choice < 0)]]
cmap = matplotlib.cm.get_cmap('bwr')
colors2 = [cmap(0), cmap(255)]
ax = plot_projections(_trials, -m, -wi1, colors2, 'Rec. output $m$', 'Input $I^A$', _trials_other)
ax.set_title('Context 1: $m$ vs $I^A$ (solid=CTRNN, dotted=fitted LR)')
plt.show()


In [ ]:
_trials = [traj[(context == 1) & (choice > 0) & (color > 0)],
           traj[(context == 1) & (choice > 0) & (color < 0)],
           traj[(context == 1) & (choice < 0) & (color > 0)],
           traj[(context == 1) & (choice < 0) & (color < 0)]]
_trials_other = [traj2[(context == 1) & (choice > 0) & (color > 0)],
                 traj2[(context == 1) & (choice > 0) & (color < 0)],
                 traj2[(context == 1) & (choice < 0) & (color > 0)],
                 traj2[(context == 1) & (choice < 0) & (color < 0)]]
cmap = matplotlib.cm.get_cmap('PiYG')
colors4 = [cmap(0), cmap(255), cmap(0), cmap(255)]
plot_projections(_trials, -m, wi2, colors4, 'Rec. output $m$', 'Input $I^B$', _trials_other)
plt.title('Context 1: $m$ vs $I^B$ (solid=CTRNN, dotted=fitted LR)')
plt.show()


In [ ]:
_trials = [traj[(context == -1) & (choice > 0)], traj[(context == -1) & (choice < 0)]]
_trials_other = [traj2[(context == -1) & (choice > 0)], traj2[(context == -1) & (choice < 0)]]
cmap = matplotlib.cm.get_cmap('PiYG')
colors2 = [cmap(0), cmap(255)]
plot_projections(_trials, -m, -wi2, colors2, 'Rec. output $m$', 'Input $I^B$', _trials_other)
plt.title('Context 2: $m$ vs $I^B$ (solid=CTRNN, dotted=fitted LR)')
plt.show()


In [ ]:
_trials = [traj[(context == -1) & (choice > 0) & (motion > 0)],
           traj[(context == -1) & (choice > 0) & (motion < 0)],
           traj[(context == -1) & (choice < 0) & (motion > 0)],
           traj[(context == -1) & (choice < 0) & (motion < 0)]]
_trials_other = [traj2[(context == -1) & (choice > 0) & (motion > 0)],
                 traj2[(context == -1) & (choice > 0) & (motion < 0)],
                 traj2[(context == -1) & (choice < 0) & (motion > 0)],
                 traj2[(context == -1) & (choice < 0) & (motion < 0)]]
cmap = matplotlib.cm.get_cmap('bwr')
colors4 = [cmap(0), cmap(255), cmap(0), cmap(255)]
plot_projections(_trials, -m, -wi1, colors4, 'Rec. output $m$', 'Input $I^A$', _trials_other)
plt.title('Context 2: $m$ vs $I^A$ (solid=CTRNN, dotted=fitted LR)')
plt.show()


### 8.2 TDR-Connectivity overlap

We compare the task-relevant axes uncovered by Targeted Dimensionality Reduction (choice,
motion, color, context) with the fitted low-rank connectivity vectors. Large overlap indicates
that the LINT-inferred vectors align with the task variables.


In [ ]:
vecs_tdr = [beta_motion, beta_color, beta_context, beta_choice]
vecs_fit = [wi1, wi2, wi_ctx1, wi_ctx2, m, n, wo]
ov = overlap_matrix2(vecs_tdr, vecs_fit, norm='l2', abs=True)

fig, ax = plt.subplots(figsize=(12, 5))
plot_ovmat(ov, cmap='abs', figsize=(12, 5), ax=ax)
ax.set_xlabel('Fitted connectivity')
ax.set_ylabel('TDR axes')
ax.set_xticks(np.arange(0.5, 7, 1))
ax.set_xticklabels(['$I^A$', '$I^B$', '$I^{ctxA}$', '$I^{ctxB}$', '$m$', '$n$', '$w$'])
ax.set_yticks(np.arange(0.5, 4, 1))
ax.set_yticklabels(['stim. A', 'stim. B', 'context', 'choice'])
ax.set_title('TDR-Connectivity overlap')
plt.tight_layout()
plt.show()


## 9. Inactivation experiments

A key prediction of the inferred low-rank model is that silencing the context-selective
populations impairs performance specifically in the corresponding context. Because LINT provides
a one-to-one mapping between neurons in the fitted network and neurons in the CTRNN teacher, we
can inactivate the *same* set of neurons in both networks and compare the behavioral effect.

We generate validation trials locked to context A or context B, measure baseline accuracy, then
remove population 0 (non-selective), population 1 (context-A selective), and population 2
(context-B selective) one at a time. We expect inactivating population 1 to hurt context-A
performance but spare context B, and vice versa for population 2. Removing the non-selective
population provides a control that should degrade performance in both contexts.


In [ ]:
x_ctxA, y_ctxA, mask_ctxA = generate_mante_data(
    100, fraction_validation_trials=0, context_spec=1, seed=101)
x_ctxB, y_ctxB, mask_ctxB = generate_mante_data(
    100, fraction_validation_trials=0, context_spec=2, seed=102)

_, acc_orig_lrA = test_mante(net_lr_final, x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_orig_lrB = test_mante(net_lr_final, x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))
print(f'LR baseline: ctx A={acc_orig_lrA:.3f}, ctx B={acc_orig_lrB:.3f}')

net_lr_final.to('cpu')
axes = psychometric_matrices(net_lr_final, colorbar=True)
axes[0].set_title('Context = Color')
axes[1].set_title('Context = Motion')
plt.suptitle('Fitted low-rank network', y=1.02)
plt.show()
net_lr_final = net_lr_final.to(device)


In [ ]:
net_inac0_lr = inactivate_pop(net_lr_final, z == 0)
net_inac1_lr = inactivate_pop(net_lr_final, z == 1)
net_inac2_lr = inactivate_pop(net_lr_final, z == 2)

_, acc_inac0_lrA = test_mante(net_inac0_lr.to(device), x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_inac0_lrB = test_mante(net_inac0_lr.to(device), x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))
_, acc_inac1_lrA = test_mante(net_inac1_lr.to(device), x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_inac1_lrB = test_mante(net_inac1_lr.to(device), x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))
_, acc_inac2_lrA = test_mante(net_inac2_lr.to(device), x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_inac2_lrB = test_mante(net_inac2_lr.to(device), x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))

print(f'Inactivate pop 0: ctx A={acc_inac0_lrA:.3f}, ctx B={acc_inac0_lrB:.3f}')
print(f'Inactivate pop 1: ctx A={acc_inac1_lrA:.3f}, ctx B={acc_inac1_lrB:.3f}')
print(f'Inactivate pop 2: ctx A={acc_inac2_lrA:.3f}, ctx B={acc_inac2_lrB:.3f}')

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.scatter([1, 2],
           [acc_orig_lrA, acc_orig_lrB],
           c='darkslategray', s=50)
ax.scatter([4, 5],
           [acc_inac0_lrA, acc_inac0_lrB],
           c='goldenrod', s=50)
ax.scatter([7, 8],
           [acc_inac1_lrA, acc_inac1_lrB],
           c='darkslategray', s=50)
ax.scatter([10, 11],
           [acc_inac2_lrA, acc_inac2_lrB],
           c='indianred', s=50)
ax.set_ylim(0, 1.05)
ax.axhline(1, ls='--', color='gray')
ax.axhline(0.5, ls='dotted', color='indianred')
ax.set_xticks([1, 2, 4, 5, 7, 8, 10, 11])
ax.set_xticklabels(['ctx A', 'ctx B'] * 4, rotation=90)
ax.set_ylabel('Task perf.')
ax.set_title('Inactivation on fitted low-rank network')
plt.tight_layout()
plt.show()


In [ ]:
net_inac0_fr = inactivate_pop(net_ctrnn, z == 0)
net_inac1_fr = inactivate_pop(net_ctrnn, z == 1)
net_inac2_fr = inactivate_pop(net_ctrnn, z == 2)

_, acc_orig_frA = test_mante(net_ctrnn, x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_orig_frB = test_mante(net_ctrnn, x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))

_, acc_inac0_frA = test_mante(net_inac0_fr.to(device), x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_inac0_frB = test_mante(net_inac0_fr.to(device), x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))
_, acc_inac1_frA = test_mante(net_inac1_fr.to(device), x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_inac1_frB = test_mante(net_inac1_fr.to(device), x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))
_, acc_inac2_frA = test_mante(net_inac2_fr.to(device), x_ctxA.to(device), y_ctxA.to(device), mask_ctxA.to(device))
_, acc_inac2_frB = test_mante(net_inac2_fr.to(device), x_ctxB.to(device), y_ctxB.to(device), mask_ctxB.to(device))

print(f'CTRNN baseline: ctx A={acc_orig_frA:.3f}, ctx B={acc_orig_frB:.3f}')
print(f'Inactivate pop 0: ctx A={acc_inac0_frA:.3f}, ctx B={acc_inac0_frB:.3f}')
print(f'Inactivate pop 1: ctx A={acc_inac1_frA:.3f}, ctx B={acc_inac1_frB:.3f}')
print(f'Inactivate pop 2: ctx A={acc_inac2_frA:.3f}, ctx B={acc_inac2_frB:.3f}')

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.scatter([1, 2],
           [acc_orig_frA, acc_orig_frB],
           c='darkslategray', s=50)
ax.scatter([4, 5],
           [acc_inac0_frA, acc_inac0_frB],
           c='goldenrod', s=50)
ax.scatter([7, 8],
           [acc_inac1_frA, acc_inac1_frB],
           c='darkslategray', s=50)
ax.scatter([10, 11],
           [acc_inac2_frA, acc_inac2_frB],
           c='indianred', s=50)
ax.set_ylim(0, 1.05)
ax.axhline(1, ls='--', color='gray')
ax.axhline(0.5, ls='dotted', color='indianred')
ax.set_xticks([1, 2, 4, 5, 7, 8, 10, 11])
ax.set_xticklabels(['ctx A', 'ctx B'] * 4, rotation=90)
ax.set_ylabel('Task perf.')
ax.set_title('Inactivation on CTRNN teacher')
plt.tight_layout()
plt.show()


## 10. Single-trial $\kappa_1$ projections

In a rank-1 network the recurrent dynamics are summarized by $\kappa_1(t) = m^\top r(t)$. We compare this projection between the CTRNN teacher and the fitted low-rank network for a few single-trial conditions.


In [ ]:
labels_cond = ['++', '+-', '-+', '--']

x1, _, _ = generate_mante_data(1, context_spec=1, coh_color_spec=1, coh_motion_spec=1, fraction_validation_trials=0, seed=201)
x2, _, _ = generate_mante_data(1, context_spec=1, coh_color_spec=-1, coh_motion_spec=1, fraction_validation_trials=0, seed=202)
x3, _, _ = generate_mante_data(1, context_spec=1, coh_color_spec=1, coh_motion_spec=-1, fraction_validation_trials=0, seed=203)
x4, _, _ = generate_mante_data(1, context_spec=1, coh_color_spec=-1, coh_motion_spec=-1, fraction_validation_trials=0, seed=204)
x = torch.cat([x1, x2, x3, x4], axis=0)

net_ctrnn.eval()
net_lr_final.eval()
with torch.no_grad():
    out_t = net_ctrnn(x.to(device), return_states=True)
    traj = out_t.states.detach().cpu().numpy()
    out_l = net_lr_final(x.to(device), return_states=True)
    traj2 = torch.tanh(out_l.states).detach().cpu().numpy()

k1 = traj @ m
k1_2 = traj2 @ m

fig, ax = plt.subplots(figsize=(6, 4))
for i in range(4):
    p, = ax.plot(k1[i], label=labels_cond[i], lw=2)
    ax.plot(k1_2[i], ls='--', c=p.get_color(), lw=2)
ax.legend()
ax.set_xlabel('Time')
ax.set_ylabel('$\kappa_1$')
ax.set_title('Context 1 (solid=CTRNN, dashed=fitted LR)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
x1, _, _ = generate_mante_data(1, context_spec=2, coh_color_spec=1, coh_motion_spec=1, fraction_validation_trials=0, seed=301)
x2, _, _ = generate_mante_data(1, context_spec=2, coh_color_spec=-1, coh_motion_spec=1, fraction_validation_trials=0, seed=302)
x3, _, _ = generate_mante_data(1, context_spec=2, coh_color_spec=1, coh_motion_spec=-1, fraction_validation_trials=0, seed=303)
x4, _, _ = generate_mante_data(1, context_spec=2, coh_color_spec=-1, coh_motion_spec=-1, fraction_validation_trials=0, seed=304)
x = torch.cat([x1, x2, x3, x4], axis=0)

with torch.no_grad():
    out_t = net_ctrnn(x.to(device), return_states=True)
    traj = out_t.states.detach().cpu().numpy()
    out_l = net_lr_final(x.to(device), return_states=True)
    traj2 = torch.tanh(out_l.states).detach().cpu().numpy()

k1 = traj @ m
k1_2 = traj2 @ m

fig, ax = plt.subplots(figsize=(6, 4))
for i in range(4):
    p, = ax.plot(k1[i], label=labels_cond[i], lw=2)
    ax.plot(k1_2[i], ls='--', c=p.get_color(), lw=2)
ax.legend()
ax.set_xlabel('Time')
ax.set_ylabel('$\kappa_1$')
ax.set_title('Context 2 (solid=CTRNN, dashed=fitted LR)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## 11. Gain Distributions

The effective gain of each neuron is the local slope of the tanh nonlinearity at its membrane
potential, $\phi'(x_i)$. Context-selective populations receive large context inputs that shift
their membrane potentials to different operating points, producing context-specific gain
modulation. We plot the population-averaged tanh curve (black) and the smoothed distributions of
mean membrane potential for context-A-selective (green) and context-B-selective (purple)
populations during the context-only period. The top row shows the fitted low-rank network; the
bottom row shows the CTRNN teacher.


In [ ]:
colors = ['dimgray', 'seagreen', 'rebeccapurple']

x_ctx1, _, _ = generate_mante_data(50, context_spec=1, fraction_validation_trials=0, seed=401)
with torch.no_grad():
    out_t = net_ctrnn(x_ctx1.to(device), return_states=True)
    traj1 = out_t.states.detach().cpu().numpy()
    out_l = net_lr_final(x_ctx1.to(device), return_dynamics=True)
    traj2 = out_l[1].detach().cpu().numpy()

x_ctx2, _, _ = generate_mante_data(50, context_spec=2, fraction_validation_trials=0, seed=402)
with torch.no_grad():
    out_t = net_ctrnn(x_ctx2.to(device), return_states=True)
    traj3 = out_t.states.detach().cpu().numpy()
    out_l = net_lr_final(x_ctx2.to(device), return_dynamics=True)
    traj4 = out_l[1].detach().cpu().numpy()

xs = np.linspace(-2.2, 2.2, 100)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_gains(traj2, z, colors, xs, 'Fitted LR, context 1', show_ylabel=True, ax=axes[0, 0])
plot_gains(traj4, z, colors, xs, 'Fitted LR, context 2', ax=axes[0, 1])
plot_gains(traj1, z, colors, xs, 'CTRNN, context 1', show_xlabel=True, show_ylabel=True, ax=axes[1, 0])
plot_gains(traj3, z, colors, xs, 'CTRNN, context 2', show_xlabel=True, ax=axes[1, 1])
plt.tight_layout()
plt.show()


## 12. Summary

This notebook showed that the NeuralRNN `Trainer` + `SupervisedObjective` pipeline is sufficient
for the full LINT workflow on a CTRNN-trained Mante task, and that it reproduces the full suite
of analyses from Valente et al. (2022) Figure 3, 4 and Supplementary Figure 2:

1. **CTRNN task training** used the same `Trainer` + `SupervisedObjective(regression)` call as any
   other supervised task.
2. **Trajectory collection** used `return_states=True` on the trained CTRNN; its hidden states are
   already post-nonlinearity firing rates.
3. **LINT fitting** used the *same* `Trainer` + `SupervisedObjective(regression)` by:
   - setting the low-rank network's `output_dim = latent_dim`,
   - fixing its readout to the identity (`wo = N * I`),
   - supplying the CTRNN firing-rate trajectories as regression targets.
4. **Readout transfer** copied the CTRNN readout into a second low-rank network with
   `output_dim=1`, recovering task accuracy.
5. **Rank scan** compared LINT with a truncated-SVD structural baseline (rank kept at 1 as requested).
6. **Connectivity analysis** included population clustering, SVD-vs-fitted / TDR-vs-fitted
   overlap matrices, and scatter plots of connectivity vectors.
7. **Dynamics analysis** included PCA cumulative variance and $R^2$ vs. kept PCs.
8. **TDR projections** showed that LINT connectivity vectors align with task variables, including
   both TDR-axis and connectivity-axis projections for both contexts.
9. **Inactivation experiments** confirmed context-specific deficits in both networks, including
   the effect of removing non-selective neurons (population 0).
10. **Single-trial $\kappa_1$ trajectories** showed close agreement between CTRNN teacher and
    fitted low-rank network in both contexts.
11. **Gain distributions** showed matching context-dependent gain modulation.

No custom training loop and no modifications to `Trainer`, `SupervisedObjective`,
`LowrankRNNModel`, or `CTRNNModel` were required.
